# Customer Segmentation (Mall Customers)

**Goal:** Group customers into segments for targeted marketing
**Algorithm:** K-Means Clustering + PCA Visualization

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
%matplotlib inline

In [ ]:
np.random.seed(42)
n = 200

customer_id = np.arange(1, n + 1)
gender = np.random.choice(['Male', 'Female'], n)
age = np.random.randint(18, 70, n)
income = np.random.randint(15, 140, n)
spending = np.random.randint(1, 100, n)

df = pd.DataFrame({
    'CustomerID': customer_id,
    'Gender': gender,
    'Age': age,
    'Annual Income (k$)': income,
    'Spending Score (1-100)': spending
})
print ('Shape: %s' % (df.shape,))
print ('First 5 rows:\n%s' % df.head())

<hr>## 1. Exploratory Data Analysis

In [ ]:
print ('Age range: %d - %d' % (df['Age'].min(), df['Age'].max()))
print ('Income range: %d - %d' % (df['Annual Income (k$)'].min(), df['Annual Income (k$)'].max()))
print ('Spending range: %d - %d' % (df['Spending Score (1-100)'].min(), df['Spending Score (1-100)'].max()))
print ('\nGender distribution:\n%s' % df['Gender'].value_counts())

<hr>## 2. Feature Selection & Scaling

In [ ]:
features = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print ('Scaled feature matrix: %s' % (X_scaled.shape,))

<hr>## 3. Find Optimal K (Elbow Method)

In [ ]:
inertias = []
K_range = range(1, 11)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertias, 'bo-')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal K')
plt.grid(True)
plt.tight_layout()
plt.show()
print ('Optimal K is where the elbow bends (usually K=5)')

<hr>## 4. Train K-Means

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)
print ('Cluster sizes:\n%s' % df['Cluster'].value_counts().sort_index())

<hr>## 5. Cluster Analysis

In [ ]:
profile = df.groupby('Cluster')[features].mean().round(1)
profile['Count'] = df['Cluster'].value_counts().sort_index().values
print ('Cluster Profiles (mean values):\n%s' % profile)

In [ ]:
print ('\nInterpretation Guide:')
print ('Cluster 0: Low income, low spending  - Budget conscious')
print ('Cluster 1: High income, low spending  - Selective')
print ('Cluster 2: Low income, high spending  - Impulsive')
print ('Cluster 3: High income, high spending - Premium (target!)')
print ('Cluster 4: Medium income, medium spending - Average')

<hr>## 6. Visualize with PCA

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
df['PCA1'] = X_pca[:, 0]
df['PCA2'] = X_pca[:, 1]

plt.figure(figsize=(10, 6))
for cluster in range(5):
    mask = df['Cluster'] == cluster
    plt.scatter(df.loc[mask, 'PCA1'], df.loc[mask, 'PCA2'],
                label='Cluster %d' % cluster, alpha=0.7)

plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.title('Customer Segments Visualized with PCA')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()